# ArgosQC SMRU Project: Config, Run QC and Upload `otnsat` DB


### Supports near real-time (NRT) and delay mode

#### NRT mode (download and run ArgosQC nightly and upload to `otnsat`.`satnrt` schema): 
1.1 Requires user and password to SMRU portal to download <cid>.mdb.


#### Delay Mode (one time ArgosQC and upload to `otnsat`.`delay` schema)
1.2 Obtain .mdb from PI and upload to specified folder.


        
2. Extract `deployments` table from SMRU or local for observation.
3. (Optional) Extract tracks (from the raw `ctd` table) for visualization
4. (Optional) Visualize Tracks (Raw Data) - Helps to Determine which Tags to be excluded.
5. Generate smru_qc Configuration Files.
6. Run ArgosQC for Configured Project.
7. Evaluate ArgosQC Resluts.
8. Visualize SSM Output with Raw Tracks - Helps to Verify and Improve Model Perforamce.
9. Load into OTNSAT DB.

In [ ]:
from py_nrt import argos_smru_project as asp
from py_nrt import load_nrt_results as lnr
from py_nrt.common import get_engine
import socket, itables

if socket.gethostname() == 'jphub.oceantrack.org':
    # Specific settings for JPhub 
    notebook_base_url = 'https://jphub.oceantrack.org/user/satnrt/tree/rt-sat-to-obis'
    mdb_tables_path = '/usr/bin'
    r_executable = '/opt/R/4.5.2/bin/Rscript'
else:
    # Must specify for your local environment
    notebook_base_url = 'http://localhost:8888/tree'
    # Excutable for mdb tables tool. Set to empty string for Windows
    mdb_tables_path = 'C:/Users/hpz2/mdbtools-app'
    r_executable = 'C:/PROGRA~1/R/R-45~1.2/bin/x64/Rscript'

if not r_executable:
    raise Exception(f'Please specify r_executable path')
if not notebook_base_url:
    raise Exception(f'Please specify notebook_base_url for your environment.')

qc_input_path = r'input'
qc_output_path = r'qc'
argosqc_r_script = 'r_nrt/run_ArgosQC_smru_qc.R'

exclude_tag_ref = []

# (Optional) Set to True to see API payload and response.
verbose=False
program_dropdown = asp.show_program_dropdown()
upload_mode_radio = asp.show_data_upload_mode_radio()
cid_textbox = asp.show_cid_textbox()
collectioncode_textbox = asp.show_collectioncode_textbox()

## 1. Get `cid`.mdb file from SMRU portal or local upload


In [ ]:
program, upload_mode, cid, collectioncode = asp.get_user_input(
    {'program': program_dropdown, 'upload_mode': upload_mode_radio, 'cid': cid_textbox, 'collectioncode': collectioncode_textbox})
smru_user_text, smru_password_text = asp.show_smru_login_or_local_mdb(qc_input_path, program, upload_mode, cid, collectioncode, notebook_base_url)

## 2. Extract `deployments` table for observation

### Requires `mdbtools` 
1. For Windows 
    - Download the `mdbtools-app` folder from https://github.com/apelsito/mdb-tools-for-windows/tree/main 
    - Save into `C:\mdbtools-app` and add to system path.
    - To test: in a `cmd` terminal type `mdb-tables --help`. Expected output:
    ```
        option parsing failed: Unrecognized option: --help.
        Usage:
          mdb-util [OPTIONa€|] <file> - show MDB files tables/entries
        ...
    ```
2. For Mac: 
    - Intall mdbtools: `brew install mdbtools`
    - It should be added into PATH. To verify: `which mdb-export`
    

In [ ]:
if smru_user_text and smru_password_text:
    asp.smru_get_mdb(program, cid, smru_user_text.value, smru_password_text.value, qc_input_path, collectioncode, 120, True)
# Extract deployments table from <cid>.mdb file.
cid_mdb, deployment_df = asp.extract_deployments(program, cid, qc_input_path, verbose=verbose)    

### 2.1 Identify invalid tag(s) to be excluded (note the REFs) from above table.
- If the preject has been create on OTN node, find matching tags in OTN project schema.otn_sat_tags and list tags to be excluded in `exclude_tag_ref` and go to Step 5.
- Otherwise, use step 3 and step 4 to determin tags to be excluded.

In [ ]:
exclude_tag_ref = []

## 3. (Optional) Extract tracks (from the raw `ctd` table) for visualization


In [ ]:
cid_mdb, tracks_df = asp.extract_tacks(program, cid, qc_input_path, exclude_tag_ref, verbose=verbose)
# (Optional) specify a sebset of tags (REF) to export. Leave subset_tags = [] to export all tags
subset_tags = []
subset_tag_tracks_df = asp.export_for_kepler(cid, tracks_df, subset_tags)

## 4. (Optional) Visualize Tracks (Raw Data) - Helps to Determine which Tags to be excluded
Follow step 1 to 4 in `Visualize Satellite Tracks on Kepler.gl` notebook

## 5. Generate smru_qc Configuration Files

- Reference to all SMRU ArgosQC Parameters: https://github.com/ianjonsen/ArgosQC/blob/main/vignettes/SMRU_config_file.Rmd

### Required Parameters:

5.1 `time_step` (hours): adjust value (usually between 1 to 10) according to tagged animal's species.

5.2 `common_name` and `species`

5.3 `release_site` and `state_country`


In [ ]:
download = False if (not smru_user_text) and (not smru_password_text) else True
user = smru_user_text.value if smru_user_text is not None else ''
password = smru_password_text.value if smru_password_text is not None else ''


time_step = 3
# Campaign specific settings
common_name = 'grey seal'
species = 'Halichoerus grypus'
release_site = 'Hay Island'
state_country = 'Nova Scotia Canada'

output_files = asp.create_smru_qc_config(
    upload_mode = upload_mode,
    program = program,
    cid = cid,
    deployment_df=deployment_df,
    drop_ids = exclude_tag_ref,
    otn_collection_code = collectioncode,
    qc_input_path = qc_input_path,
    qc_output_path=qc_output_path,
    mdb_tables_path=mdb_tables_path,
    user=user, 
    password=password, 
    time_step=time_step,
    download=download,
    common_name=common_name,
    species=species,
    release_site=release_site,
    state_country=state_country,
    notebook_base_url=notebook_base_url
)
# TODO: Add links to config parameters 

## 6. Run ArgosQC for Configured Project

In [ ]:
asp.run_smru_qc(r_executable, str(output_files[0]), argosqc_r_script = argosqc_r_script)

## 7. Evaluate ArgosQC Resluts

### Run below cell and open QC results link to review images under `diag` and `maps` for model fitting.

### Download and review ArgosQC reslut: `ssmoutputs_<cid>_nrt.csv`.

In [ ]:
asp.show_argosqc_results(qc_output_path, notebook_base_url, program, cid)
subset_tags = [ '']
tag_ssmoutput_df = asp.extract_ssmoutput_tracks(program, cid, qc_output_path, subset_tags, verbose=verbose)

## 8. Visualize SSM Output with Raw Tracks - Helps to Verify and Improve Model Perforamce

Follow step 5 in Visualize Satellite Tracks on Kepler.gl notebook


## 9. Load into OTNSAT DB

- Provide OTNSAT DB auth.

In [ ]:
sat_delay_schema = 'test'
engine = get_engine(r"py_nrt/database_conn_string.auth")
lnr.check_sat_db_backend(engine, sat_delay_schema)

In [ ]:
ssmoutput_last_modified_map = lnr.get_campaign_qc_results_for_program(qc_output_path, program)
ssmoutput_last_modified_map

In [ ]:
summary_df, _ = lnr.load_to_nrt_db(engine, ssmoutput_last_modified_map)
if summary_df is not None and not summary_df.empty:
    print('Load summary by tag.')
    itables.show(summary_df)